This variation of BC+PPO is strictly based on the paper and implementation of "Bootstrapping Reinforcement Learning with Imitation for Vision-Based Agile Flight" by UZH.
Kindly refer to the paper at: https://arxiv.org/pdf/2403.12203

In [81]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3 import PPO
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical


In [82]:
class ActrorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic= nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def get_value(self, x):
            return self.critic(x)
        
    def get_action_value(self, x, action = None):
            logits = self.actor(x)
            probs = Categorical(logits = logits)
            if action is None:
                action = probs.sample()
            return action, probs.log_prob(action), probs.entropy(), self.critic(x)

In [83]:
def Expert(state):
    a = state[2]
    b = state[3]
    return 1 if (a + 0.25 * b) > 0 else 0

In [84]:
def get_expert_data(env, replay_buffer, num_tuples = 10_000):
    state, _ = env.reset()
    for _ in range(num_tuples):
        action = Expert(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        action_array = np.array([action])
        reward_array = np.array([reward])
        done_array = np.array([terminated or truncated], dtype = np.float32)
        replay_buffer.add(
            obs = state,
            next_obs = next_state,
            action = action_array,
            reward = reward_array,
            done = done_array,
            infos =[info]
        )
        if terminated or truncated:
            state, _ = env.reset()
        else:
            state = next_state

In [85]:
def warm_start(agent, replay_buffer, lr = 1e-4, epochs = 20, batch_size = 64, num_samples = 2000):
    print(f"Starting the Warm-Up now (for {epochs} epochs)")
    optimizer = optim.Adam(agent.actor.parameters(), lr = lr)
    loss_fn = nn.CrossEntropyLoss()
    iterations_per_epoch = num_samples // batch_size
    total_steps = epochs*iterations_per_epoch

    for step in range(total_steps):
        samples = replay_buffer.sample(batch_size = batch_size)
        batch_states = samples.observations
        batch_actions = samples.actions.flatten().long()
        logits = agent.actor(batch_states)
        loss = loss_fn(logits, batch_actions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step%iterations_per_epoch == 0:
            current_epoch = step // iterations_per_epoch+1
    print('Warm up done')

In [86]:
def train_PPO(env, agent, total_timesteps = 1_000_000, num_steps = 2048, gae_lambda = 0.95, gamma = 0.99, ppo_epochs = 25, mini_batch_size = 64, clip_coeff = 0.2, ent_coeff = 0.001, vf_coeff = 0.5, lr = 1e-4):

    optimizer = optim.Adam(agent.parameters(), lr = lr)
    states = torch.zeros((num_steps, *env.observation_space.shape))
    actions = torch.zeros(num_steps)
    logprobs = torch.zeros(num_steps)
    rewards = torch.zeros(num_steps)
    dones = torch.zeros(num_steps)
    values = torch.zeros(num_steps)

    next_state, _ = env.reset()
    next_state = torch.tensor(next_state, dtype=torch.float32)
    next_done = torch.tensor(0.0)
    global_step = 0

    while global_step < total_timesteps:
        for step in range(num_steps):
            global_step += 1
            states[step] = next_state
            dones[step] = next_done
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_value(next_state)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob

            next_state_np, reward, terminated,truncated, _ = env.step(action.item())
            rewards[step] = torch.tensor(reward, dtype = torch.float32)
            next_done = torch.tensor(float(terminated or truncated))
            next_state = torch.tensor(next_state_np, dtype = torch.float32)
            if terminated or truncated:
                next_state_np, _ = env.reset()
                next_state = torch.tensor(next_state_np, dtype = torch.float32)
        
        with torch.no_grad():
            next_value = agent.get_value(next_state).flatten()
            advantages = torch.zeros_like(rewards)
            last_gae = 0
            for t in reversed(range(num_steps)):
                if t == num_steps -1:
                    next_non_terminal = 1 - next_done
                    next_values = next_value
                else:
                    next_non_terminal = 1 - dones[t +1]
                    next_values = values[t+1]
                delta = rewards[t] + gamma * next_values * next_non_terminal - values[t]
                advantages[t] = last_gae = delta+ gamma * gae_lambda * next_non_terminal * last_gae
            returns = advantages + values

        b_states = states
        b_actions = actions
        b_logprobs = logprobs
        b_advantages = advantages
        b_returns = returns

        b_inds = np.arange(num_steps)
        for epoch in range(ppo_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, num_steps, mini_batch_size):
                end = start+mini_batch_size
                mb_inds = b_inds[start:end]
                _, newlogprob, entropy, newvalues = agent.get_action_value(b_states[mb_inds], b_actions[mb_inds].long())
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                mb_advantages = b_advantages[mb_inds]
                mb_advantages = (mb_advantages - mb_advantages.mean())/(mb_advantages.std() + 1e-10)
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages* torch.clamp(ratio, 1-clip_coeff, 1+ clip_coeff)
                actor_loss = torch.max(pg_loss1, pg_loss2).mean()
                critic_loss = 0.5 * ((newvalues.flatten() - b_returns[mb_inds])**2).mean()
                loss = actor_loss + vf_coeff*critic_loss - ent_coeff*entropy.mean()

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), max_norm=0.5)
                optimizer.step()
        print(f"Step: {global_step}/{total_timesteps} | Actor Loss: {actor_loss.item():.4f} | Critic Loss: {critic_loss.item():.4f}")

In [87]:
if __name__ == '__main__':
    ENV_ID = 'CartPole-v1'
    Num_Expert_Samples = 10000
    training_env = gym.make(ENV_ID)
    ppo_agent = ActrorCritic(state_dim = training_env.observation_space.shape[0], action_dim=training_env.action_space.n)
    expert_buffer = ReplayBuffer(buffer_size= Num_Expert_Samples, observation_space =training_env.observation_space, action_space=training_env.action_space, device='cpu', n_envs = 1)
    get_expert_data(training_env, expert_buffer, num_tuples=Num_Expert_Samples)
    warm_start(ppo_agent, expert_buffer, epochs = 10_000, num_samples=Num_Expert_Samples)
    train_PPO(training_env, ppo_agent, total_timesteps=100_000)
    training_env.close()

Starting the Warm-Up now (for 10000 epochs)
Warm up done
Step: 2048/100000 | Actor Loss: 0.0056 | Critic Loss: 4.4455
Step: 4096/100000 | Actor Loss: -0.0058 | Critic Loss: 8.3803
Step: 6144/100000 | Actor Loss: 0.0035 | Critic Loss: 18.0518
Step: 8192/100000 | Actor Loss: -0.0216 | Critic Loss: 12.7270
Step: 10240/100000 | Actor Loss: 0.0001 | Critic Loss: 9.5770
Step: 12288/100000 | Actor Loss: 0.0218 | Critic Loss: 4.8307
Step: 14336/100000 | Actor Loss: 0.0073 | Critic Loss: 21.9084
Step: 16384/100000 | Actor Loss: -0.0146 | Critic Loss: 71.8741
Step: 18432/100000 | Actor Loss: -0.0001 | Critic Loss: 12.2855
Step: 20480/100000 | Actor Loss: -0.0128 | Critic Loss: 12.4546
Step: 22528/100000 | Actor Loss: -0.0002 | Critic Loss: 7.5997
Step: 24576/100000 | Actor Loss: 0.0018 | Critic Loss: 16.6367
Step: 26624/100000 | Actor Loss: -0.0008 | Critic Loss: 11.9845
Step: 28672/100000 | Actor Loss: 0.0020 | Critic Loss: 33.7769
Step: 30720/100000 | Actor Loss: -0.0004 | Critic Loss: 34.1686